# Episode 30: GRPO in Hugging Face TRL on our own small GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep030-trl.ipynb)

We train our own GPT with GRPO in Hugging Face's TRL, set up like the GRPO we wrote by hand and then with TRL's defaults, and measure both against our own.

[The video](https://youtu.be/MalYv-Ov9N0) · [The episode's page](https://mvgpt.nenuadrian.com/ep030-trl/) · [The docs](https://nenuadrian.github.io/mvGPT/trl/) · [The code at `ep030/end`](https://github.com/nenuadrian/mvGPT/tree/ep030/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep030/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("sft")

## The way into TRL, and back
Our GPT saved as a GPT-2, loaded back into our model, and checked weight by weight and on its
logits.

In [ ]:
!python -m pytest -q tests/test_hub.py

## The data

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids

## GRPO in TRL, with our settings
Episode 16's GRPO, set up in TRL's `GRPOConfig`: 16 instructions and 8 stories a rollout, two
passes, a KL weight of 0.05, float32. 32 rollouts, to `checkpoints/trl/matched`.

In [ ]:
!python trl_grpo.py 32 matched

## With TRL's defaults
No KL penalty and no reference model, the loss over all the batch's tokens, one pass, bfloat16
and gradient checkpointing. To `checkpoints/trl/defaults`.

In [ ]:
!python trl_grpo.py 32 defaults

## Against SFT, and stories

In [ ]:
!python evaluate.py checkpoints/sft/model.pt checkpoints/trl/matched/model.pt checkpoints/trl/defaults/model.pt --out results/trl
!python sample.py checkpoints/trl/matched/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3
!python sample.py checkpoints/trl/defaults/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3